# NB03 — Baselines: the reproduction gate

**Project:** CardioMamba-Net · **Stage:** 3 of 5
`01_verify` → `02_preprocess` → **`03_baselines`** → `04_cardiomamba_train` → `05_evaluate`

---

## Why this notebook exists

Before we are allowed to claim a new architecture wins, we have to show our pipeline can
**reproduce the numbers we are trying to beat**. This notebook re-implements all four networks
from Chowdhury et al. 2024 — FPN-1D, UNet-1D, LinkNet-1D and MultiResLinkNet — trains them on our
data with their objective, and puts our numbers next to theirs.

If MultiResLinkNet does not land near a temporal correlation of **61.9** on RVA combined, something
in our pipeline is wrong and **nothing after this point is trustworthy**. That is the gate.

Two honest differences from their setup, both declared in the paper:

- **Splits are strictly by subject** and test windows do not overlap (see NB02 §"leakage decision").
  Theirs almost certainly leaked. So our reproduction may land *below* their published figures —
  that is the expected direction, and it is the fair comparison for everything that follows.
- **ECG target is [−1, 1]**, not [0, 1]. Set `CFG["TARGET_01"] = True` to run their convention.

Everything else is theirs: **1 input channel** (the arctangent-demodulated displacement `dy`),
5 levels, 64 filters doubling, **plain MSE**, Adam at 5e-4, 1024-sample windows at 128 Hz.

---

## ⚠️ Accelerator: **GPU T4 × 2**

*Session options → Accelerator → **GPU T4 x2***, Internet **On**, `HF_TOKEN` secret attached.

Both GPUs are used through `DataParallel`. AMP (mixed precision) is on, which roughly doubles
throughput on T4s and halves memory.

## The run queue — this is how it survives Kaggle

There are **4 models × 4 experiment settings × 5 folds = 80 runs**. That does not fit in one
12-hour session, and it is not supposed to.

The notebook builds a **queue**, checks which runs are already finished on Hugging Face, and works
through as many as fit in `TIME_BUDGET_H`. Then it pushes and stops cleanly. **Start a new session
and run it again** — it picks up exactly where it left off. Three or four sessions completes the
matrix. Nothing is ever recomputed.

Start with `QUICK = True`: one fold, 25 epochs, ~40 minutes, and it proves the whole path end to
end. Then set it `False` and let the queue run.

---
# 1 · Configuration

In [ ]:
CFG = {
    "SRC_REPO":  "Shanmuk4622/cr-rvs-radar-ecg-processed",   # NB02 output
    "DST_REPO":  "Shanmuk4622/cardiomamba-net",              # models + results (public)
    "HF_PRIVATE": False,
    "RUN_ID":    "nb03_baselines_v1",

    "WORK":    "/kaggle/working/nb03",
    "SCRATCH": "/kaggle/temp/nb03",
    "PUSH_INTERVAL_S": 30 * 60,
    "HF_MAX_REQ_HOUR": 120,

    # ---- faithful reproduction of Chowdhury et al. 2024 ----------------------
    "CHANNELS":   ["dy"],        # THEIR input: one arctangent-demodulated displacement channel
    "BASE":       64,            # section 3.1: "initial layer containing 64 filters"
    "LEVELS":     4,             # 5 levels counting the bottleneck
    "LR":         5e-4,          # section 3.1
    "LOSS":       "mse",         # section 3.1: "As a loss function, the MSE function was used"
    "TARGET_01":  False,         # True = their [0,1] ECG convention

    # ---- training ------------------------------------------------------------
    "EPOCHS":     120,
    "PATIENCE":   20,            # section 3.1
    "BATCH":      64,
    "WORKERS":    2,
    "WEIGHT_DECAY": 1e-4,
    "AMP":        True,
    "MULTI_GPU":  True,
    "SEED":       1337,

    # ---- the queue -----------------------------------------------------------
    "MODELS":      ["fpn", "unet", "linknet", "multireslinknet"],
    "EXPERIMENTS": ["B_rva", "A_resting", "A_valsalva", "A_apnea"],   # B first: it is the headline
    "N_FOLDS":     5,
    "TIME_BUDGET_H": 10.5,       # stop cleanly before Kaggle's 12 h wall
    "QUICK":       True,         # <-- first run: 1 fold, 25 epochs. Then set False.
    "QUICK_EPOCHS": 25,
    "QUICK_FOLDS":  1,
}
import json
print(json.dumps(CFG, indent=2))

In [ ]:
import os, sys, gc, json, math, time, warnings, subprocess, platform, shutil
from pathlib import Path
from datetime import datetime, timezone
warnings.filterwarnings("ignore")

def _pip(*p):
    import importlib.util
    miss = [x for x in p if importlib.util.find_spec(x.replace("-", "_")) is None]
    if miss:
        print("installing:", miss)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *miss], check=False)
_pip("pyarrow", "huggingface_hub")

import numpy as np, pandas as pd, torch
WORK = Path(CFG["WORK"]); SCRATCH = Path(CFG["SCRATCH"])
for d in (WORK, SCRATCH, WORK / "runs", WORK / "results", WORK / "figures"):
    d.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(WORK))

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU{i}: {p.name}  {p.total_memory/2**30:.1f} GB  sm_{p.major}{p.minor}")
    torch.backends.cudnn.benchmark = True
else:
    print("  !! NO GPU -- set Accelerator to 'GPU T4 x2' in Session options.")
    print("     The notebook will still run on CPU but training will be impractically slow.")

---
# 2 · Library

Six modules, written to disk and imported. Five come straight from NB02's repo contract
(`crvs_sync`, `crvs_data`, `crvs_metrics`) or are defined here and reused by NB04 and NB05
(`crvs_models`, `crvs_losses`, `crvs_engine`).

`crvs_models.py` is the interesting one — it holds the four baseline architectures, including the
MultiRes block and ResPath that make MultiResLinkNet what it is.

In [ ]:
MODULES = {
 "crvs_sync.py":   r"""
# crvs_sync.py -- resumable, rate-limited, interrupt-safe Hugging Face folder sync.
# Identical across NB01-NB05 so the cadence rules are enforced in exactly one place.
#   * push at most once per PUSH_INTERVAL_S (default 30 min)
#   * push immediately when a stage finishes            -> sync.stage_done("name")
#   * push immediately when execution is stopped        -> SIGINT / SIGTERM / atexit
#   * one upload_folder call per flush, behind a token bucket, backing off on 429
#   * resume by pulling the run folder back on startup
import os, json, time, random, threading, atexit, signal
from pathlib import Path
from datetime import datetime, timezone

class TokenBucket:
    # capacity = requests per hour, refilled continuously
    def __init__(self, per_hour=120):
        self.capacity = float(per_hour); self.tokens = float(per_hour)
        self.rate = per_hour / 3600.0; self.t = time.monotonic()
        self.lock = threading.Lock()
    def take(self, n=1, block=True, timeout=1200):
        deadline = time.monotonic() + timeout
        while True:
            with self.lock:
                now = time.monotonic()
                self.tokens = min(self.capacity, self.tokens + (now - self.t) * self.rate)
                self.t = now
                if self.tokens >= n:
                    self.tokens -= n; return True
                need = (n - self.tokens) / self.rate
            if not block or time.monotonic() + need > deadline:
                return False
            time.sleep(min(need, 5.0))

class HFSync:
    def __init__(self, repo_id, local_dir, token, repo_type="dataset", private=False,
                 run_id="run", push_interval_s=1800, max_req_hour=120, retry_max=6,
                 verbose=True):
        from huggingface_hub import HfApi
        self.api = HfApi(token=token); self.token = token
        self.repo_id = repo_id; self.repo_type = repo_type; self.private = private
        self.run_id = run_id
        self.local = Path(local_dir); self.local.mkdir(parents=True, exist_ok=True)
        self.interval = push_interval_s
        self.bucket = TokenBucket(max_req_hour)
        self.retry_max = retry_max; self.verbose = verbose
        self._last_push = 0.0
        self._flag = threading.Event(); self._stop = threading.Event()
        self._lock = threading.Lock()
        self._pushes = 0; self._failures = 0
        self.history = self.local / "history.jsonl"
        self.state_path = self.local / "state.json"
        self._ensure_repo(); self._install_handlers()
        self._thread = threading.Thread(target=self._loop, daemon=True, name="hf-uploader")
        self._thread.start()
        self.log("sync_started", repo=self.repo_id, private=self.private)

    def _ensure_repo(self):
        from huggingface_hub import create_repo
        create_repo(self.repo_id, repo_type=self.repo_type, private=self.private,
                    exist_ok=True, token=self.token)
        if not self.private:
            try:
                self.api.update_repo_visibility(self.repo_id, private=False,
                                                repo_type=self.repo_type, token=self.token)
            except Exception:
                pass

    @property
    def url(self):
        kind = "datasets/" if self.repo_type == "dataset" else ""
        return "https://huggingface.co/" + kind + self.repo_id

    def log(self, event, **kw):
        rec = {"ts": datetime.now(timezone.utc).isoformat(), "run": self.run_id, "event": event}
        rec.update(kw)
        try:
            with open(self.history, "a") as f:
                f.write(json.dumps(rec, default=str) + "\n")
        except Exception:
            pass
        if self.verbose and event not in ("heartbeat",):
            print("  [" + event + "] " + " ".join(f"{k}={v}" for k, v in kw.items()))

    def save_state(self, state):
        tmp = self.state_path.with_suffix(".tmp")
        tmp.write_text(json.dumps(state, indent=2, default=str)); tmp.replace(self.state_path)

    def load_state(self, default=None):
        if self.state_path.exists():
            try:
                return json.loads(self.state_path.read_text())
            except Exception:
                pass
        return default if default is not None else {}

    def pull(self, allow_patterns=None, into=None):
        from huggingface_hub import snapshot_download
        try:
            self.bucket.take(1)
            p = snapshot_download(self.repo_id, repo_type=self.repo_type, token=self.token,
                                  local_dir=str(into or self.local),
                                  allow_patterns=allow_patterns)
            self.log("resume_pull_ok", path=str(p)); return True
        except Exception as e:
            self.log("resume_pull_empty", err=type(e).__name__); return False

    def stage_done(self, name, **kw):
        self.log("stage_done", stage=name, **kw); self._flag.set()

    def _do_upload(self, msg):
        from huggingface_hub import upload_folder
        for attempt in range(self.retry_max):
            if not self.bucket.take(1, block=True, timeout=1800):
                self.log("rate_limited_giveup"); return False
            try:
                upload_folder(folder_path=str(self.local), repo_id=self.repo_id,
                              repo_type=self.repo_type, token=self.token,
                              commit_message=msg,
                              ignore_patterns=["*.tmp", "**/__pycache__/**", ".git*",
                                               "*.lock", ".cache/**"])
                self._pushes += 1; self._last_push = time.time()
                self.log("push_ok", n=self._pushes, msg=msg); return True
            except Exception as e:
                self._failures += 1
                wait = min(300, (2 ** attempt) * 5) * (0.7 + 0.6 * random.random())
                self.log("push_retry", attempt=attempt + 1,
                         err=f"{type(e).__name__}: {e}", sleep=round(wait, 1))
                time.sleep(wait)
        self.log("push_failed_permanently", msg=msg); return False

    def flush(self, final=False, msg=None):
        with self._lock:
            stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M")
            m = msg or ((self.run_id + " final") if final else (self.run_id + " @ " + stamp + "Z"))
            ok = self._do_upload(m); self._flag.clear(); return ok

    def _loop(self):
        while not self._stop.is_set():
            self._stop.wait(20)
            if self._stop.is_set():
                break
            due = (time.time() - self._last_push) >= self.interval
            want = self._flag.is_set()
            if due or want:
                try:
                    tag = "stage" if want else "periodic"
                    self.flush(msg=self.run_id + " " + tag + " @ " +
                               datetime.now(timezone.utc).strftime("%H:%M") + "Z")
                except Exception as e:
                    self.log("loop_error", err=str(e))

    def _install_handlers(self):
        def handler(signum, frame):
            self.log("interrupt", signal=int(signum))
            try:
                self.flush(final=True, msg=self.run_id + " interrupted (sig " + str(signum) + ")")
            finally:
                if signum == signal.SIGINT:
                    raise KeyboardInterrupt
        for sig in (signal.SIGINT, signal.SIGTERM):
            try:
                signal.signal(sig, handler)
            except Exception:
                pass
        atexit.register(self.close)

    def close(self):
        if self._stop.is_set():
            return
        self.log("closing"); self._stop.set()
        try:
            self.flush(final=True)
        except Exception:
            pass
""",
 "crvs_data.py":   r"""
# crvs_data.py -- windowing, folds, normalisation and the torch Dataset.
# Shared by NB03, NB04 and NB05 so every experiment sees byte-identical inputs.
import json, math
import numpy as np
from pathlib import Path

CHANNELS = ["I", "Q", "phi", "dy", "vel", "acc", "amp", "cardiac"]
# One recording is stored as a single UNCOMPRESSED .npy of shape (len(ARRAY_ROWS), n).
# It has to be .npy, not .npz: np.load(..., mmap_mode="r") silently IGNORES mmap_mode on an
# .npz, so every __getitem__ would decompress all 11 arrays to slice 1024 samples out of
# each -- measured at 23 ms per window, which would dominate the GPU time on Kaggle.
ARRAY_ROWS = CHANNELS + ["ecg_norm", "peak_map", "rr_ms"]
ROW = {name: i for i, name in enumerate(ARRAY_ROWS)}
FS       = 128
# Bumped whenever this module changes in a way the notebooks depend on. Every notebook
# asserts it after import, because writing a .py and importing it is NOT idempotent inside
# one kernel: Python caches the module in sys.modules, so a second run silently keeps the
# first version. That is how a stale .npz loader survived a rebuilt notebook once already.
LIB_VERSION = 3
WINDOW   = 1024          # 8.0 s, frozen to Chowdhury et al. 2024 section 2.3.4
HOP_TRAIN = 512          # 50 % overlap on train only
SCENARIOS = ["Resting", "Valsalva", "Apnea", "Tilt-up", "Tilt-down"]

def canon_scenario(s):
    s = str(s).strip().lower()
    for key, out in [("tiltdown", "Tilt-down"), ("tilt_down", "Tilt-down"), ("tilt-down", "Tilt-down"),
                     ("tiltup", "Tilt-up"), ("tilt_up", "Tilt-up"), ("tilt-up", "Tilt-up"),
                     ("valsalva", "Valsalva"), ("apnea", "Apnea"), ("apnoea", "Apnea"),
                     ("rest", "Resting")]:
        if key in s:
            return out
    return str(s)

def range_normalise(x, eps=1e-8):
    # z-score then squash to [-1, 1]; the baseline used [0, 1], we declare the change
    x = np.asarray(x, np.float32)
    sd = float(x.std())
    if not np.isfinite(sd) or sd < eps:
        return np.zeros_like(x, np.float32)          # constant input -> 0, not -1
    x = (x - x.mean()) / (sd + eps)
    lo, hi = np.percentile(x, 0.5), np.percentile(x, 99.5)
    x = np.clip(x, lo, hi)
    rng = float(hi - lo)
    if rng < eps:
        return np.zeros_like(x, np.float32)
    return (2.0 * (x - lo) / rng - 1.0).astype(np.float32)

def peak_heatmap(n, peaks, sigma=3.0):
    # Gaussian bumps at each R peak -- the target for the multi-task peak head
    y = np.zeros(n, np.float32)
    if len(peaks) == 0:
        return y
    half = int(math.ceil(3 * sigma))
    g = np.exp(-0.5 * (np.arange(-half, half + 1) / sigma) ** 2).astype(np.float32)
    for p in np.asarray(peaks, int):
        a, b = max(0, p - half), min(n, p + half + 1)
        y[a:b] = np.maximum(y[a:b], g[a - (p - half): (b - (p - half))])
    return y

def rr_curve(n, peaks, fs=FS, lo_ms=300.0, hi_ms=2000.0):
    # per-sample instantaneous RR interval in ms, linearly interpolated between beats
    out = np.full(n, np.nan, np.float32)
    p = np.asarray(peaks, int)
    if len(p) < 3:
        return np.nan_to_num(out, nan=800.0)
    rr = np.diff(p) / fs * 1000.0
    mid = (p[:-1] + p[1:]) / 2.0
    ok = (rr > lo_ms) & (rr < hi_ms)
    if ok.sum() < 2:
        return np.nan_to_num(out, nan=float(np.median(rr)))
    out = np.interp(np.arange(n), mid[ok], rr[ok]).astype(np.float32)
    return out

_SLOW_WARNED = {"npz": False}

class _Rec:
    # Reads one recording in whichever format is on disk.
    #   .npy (preferred) -- uncompressed, genuinely memory-mapped, ~0.3 ms per window
    #   .npz (legacy)    -- what an earlier NB02 wrote; correct but ~85x slower, because
    #                       np.load ignores mmap_mode on a zip archive and every window
    #                       decompresses all 11 arrays.
    # Both are supported so an existing corpus keeps working without a 400 MB re-upload.
    __slots__ = ("data", "kind")

    def __init__(self, rec_dir, rid):
        d = Path(rec_dir)
        pnpy, pnpz = d / (rid + ".npy"), d / (rid + ".npz")
        if pnpy.exists():
            self.data = np.load(pnpy, mmap_mode="r"); self.kind = "npy"
        elif pnpz.exists():
            self.data = np.load(pnpz); self.kind = "npz"
            if not _SLOW_WARNED["npz"]:
                _SLOW_WARNED["npz"] = True
                print("  note: reading legacy .npz recordings. Correct, but about 85x slower "
                      "per window than .npy -- re-run NB02 to regenerate the corpus and cut "
                      "the data-loading cost.")
        else:
            raise FileNotFoundError(
                f"no recording for '{rid}' in {d} (looked for .npy and .npz). "
                "Either NB02 did not finish, or the snapshot_download allow_patterns in "
                "this notebook do not cover the format NB02 wrote.")

    def rows(self, names, s, e):
        if self.kind == "npy":
            return np.array(self.data[[ROW[n] for n in names], s:e], np.float32)
        return np.stack([np.array(self.data[n][s:e], np.float32) for n in names], 0)

    def one(self, name, s, e):
        if self.kind == "npy":
            return np.array(self.data[ROW[name], s:e], np.float32)
        return np.array(self.data[name][s:e], np.float32)

class WindowDataset:
    # Slices windows on the fly, so changing WINDOW or the overlap never requires
    # re-running NB02.
    def __init__(self, rec_dir, index, norm=None, channels=None, augment=False, seed=0):
        self.rec_dir = Path(rec_dir)
        self.index = index.reset_index(drop=True)
        self.norm = norm
        self.channels = channels or CHANNELS
        self.rows = [ROW[c] for c in self.channels]
        self.augment = augment
        self.rng = np.random.RandomState(seed)
        self._cache = {}

    def __len__(self):
        return len(self.index)

    def _rec(self, rid):
        if rid not in self._cache:
            if len(self._cache) > 48:
                self._cache.pop(next(iter(self._cache)))
            self._cache[rid] = _Rec(self.rec_dir, rid)
        return self._cache[rid]

    def __getitem__(self, i):
        import torch
        r = self.index.iloc[i]
        z = self._rec(r["rec_id"])
        s, e = int(r["start"]), int(r["start"]) + WINDOW
        # _Rec.rows / _Rec.one always np.array (copy), never a view into a read-only
        # memmap -- torch.from_numpy on a non-writable array is undefined behaviour.
        x = z.rows(self.channels, s, e)
        if self.norm is not None:
            mu = np.asarray(self.norm["mean"], np.float32)[:, None]
            sd = np.asarray(self.norm["std"], np.float32)[:, None]
            x = (x - mu) / (sd + 1e-6)
        x = np.clip(x, -8.0, 8.0)
        y  = z.one("ecg_norm", s, e)
        pk = z.one("peak_map", s, e)
        rr = z.one("rr_ms", s, e) / 1000.0                           # seconds, O(1) scale
        if self.augment:
            if self.rng.rand() < 0.5:
                x = x + self.rng.randn(*x.shape).astype(np.float32) * 0.01
            if self.rng.rand() < 0.3:
                g = np.float32(1.0 + 0.1 * self.rng.randn())
                x = x * g
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(y)[None, :],
                torch.from_numpy(pk)[None, :],
                torch.from_numpy(rr)[None, :])

def compute_norm(rec_dir, index, channels=CHANNELS, max_windows=4000, seed=0):
    # Per-channel mean/std computed on TRAIN WINDOWS ONLY. Computing them over the whole
    # corpus is a classic, invisible source of leakage.
    rng = np.random.RandomState(seed)
    idx = index if len(index) <= max_windows else index.iloc[
        rng.choice(len(index), max_windows, replace=False)]
    n = 0
    s1 = np.zeros(len(channels), np.float64)
    s2 = np.zeros(len(channels), np.float64)
    cache = {}
    rec_dir = Path(rec_dir)
    for _, r in idx.iterrows():
        rid = r["rec_id"]
        if rid not in cache:
            if len(cache) > 48:
                cache.pop(next(iter(cache)))
            cache[rid] = _Rec(rec_dir, rid)
        a, b = int(r["start"]), int(r["start"]) + WINDOW
        x = cache[rid].rows(list(channels), a, b).astype(np.float64)
        s1 += x.sum(1); s2 += (x * x).sum(1); n += x.shape[1]
    mean = s1 / max(n, 1)
    var = np.maximum(s2 / max(n, 1) - mean ** 2, 1e-12)
    return {"mean": mean.tolist(), "std": np.sqrt(var).tolist(),
            "n_samples": int(n), "channels": list(channels)}
""",
 "crvs_metrics.py": r"""
# crvs_metrics.py -- every metric the baseline reports, plus the ones it should have.
import numpy as np
from scipy import signal as ss
from scipy import stats as sstats

def _f(x):
    return np.nan_to_num(np.asarray(x, np.float64), nan=0.0, posinf=0.0, neginf=0.0)

def pearson(a, b):
    a, b = _f(a), _f(b)
    if a.std() < 1e-12 or b.std() < 1e-12:
        return 0.0
    return float(np.corrcoef(a, b)[0, 1])

def psd(x, fs=128, nperseg=256):
    f, p = ss.welch(_f(x), fs=fs, nperseg=min(nperseg, len(x)))
    return f, p

def seg_metrics(y, yhat, fs=128):
    # One window. Correlations are reported x100 to match the baseline's tables.
    y, yhat = _f(y), _f(yhat)
    mae = float(np.mean(np.abs(y - yhat)))
    mse = float(np.mean((y - yhat) ** 2))
    cct = 100.0 * pearson(y, yhat)
    _, py = psd(y, fs); _, ph = psd(yhat, fs)
    ccs = 100.0 * pearson(py, ph)
    rms = lambda v: float(np.sqrt(np.mean(np.asarray(v, np.float64) ** 2)))
    rr_t = rms(yhat - y) / (rms(y) + 1e-12)
    rr_s = rms(ph - py) / (rms(py) + 1e-12)
    return {"MAE": mae, "MSE": mse, "CC_temporal": cct, "CC_spectral": ccs,
            "RRMSE_temporal": rr_t, "RRMSE_spectral": rr_s,
            "R2": float(1.0 - np.sum((y - yhat) ** 2) / (np.sum((y - y.mean()) ** 2) + 1e-12))}

def detect_r_peaks(x, fs=128, refractory_s=0.25):
    x = _f(x)
    if len(x) < int(2 * fs):
        return np.array([], int)
    ny = fs / 2.0
    sos = ss.butter(4, [5.0 / ny, min(25.0, ny * 0.95) / ny], btype="band", output="sos")
    b = ss.sosfiltfilt(sos, x)
    e = np.convolve(np.diff(b, prepend=b[0]) ** 2,
                    np.ones(max(1, int(0.10 * fs))) / max(1, int(0.10 * fs)), "same")
    thr = np.percentile(e, 98) * 0.35
    pk, _ = ss.find_peaks(e, height=thr, distance=max(1, int(refractory_s * fs)))
    return pk

def hrv_from_peaks(pk, fs=128):
    out = {"n_peaks": int(len(pk)), "mean_rr_ms": np.nan, "sd_rr_ms": np.nan,
           "mean_hr_bpm": np.nan, "sd_hr_bpm": np.nan, "rmssd_ms": np.nan}
    if len(pk) < 4:
        return out
    rr = np.diff(np.asarray(pk, float)) / fs * 1000.0
    rr = rr[(rr > 300) & (rr < 2000)]
    if len(rr) < 3:
        return out
    hr = 60000.0 / rr
    out.update(mean_rr_ms=float(rr.mean()), sd_rr_ms=float(rr.std()),
               mean_hr_bpm=float(hr.mean()), sd_hr_bpm=float(hr.std()),
               rmssd_ms=float(np.sqrt(np.mean(np.diff(rr) ** 2))))
    return out

def peak_detection_scores(y, yhat, fs=128, tol_ms=100.0):
    # Match predicted R peaks to ground-truth peaks within a tolerance window.
    gt = detect_r_peaks(y, fs); pr = detect_r_peaks(yhat, fs)
    tol = tol_ms / 1000.0 * fs
    used = np.zeros(len(pr), bool)
    tp = 0
    errs = []
    for g in gt:
        if len(pr) == 0:
            break
        # float, not the int64 that find_peaks returns -- assigning np.inf into an
        # integer array raises OverflowError even when the mask selects nothing.
        d = np.abs(pr - g).astype(np.float64)
        d[used] = np.inf
        j = int(np.argmin(d))
        if d[j] <= tol:
            tp += 1; used[j] = True; errs.append((pr[j] - g) / fs * 1000.0)
    fp = int((~used).sum()); fn = int(len(gt) - tp)
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-12)
    return {"TP": tp, "FP": fp, "FN": fn, "precision": prec, "recall": rec, "F1": f1,
            "accuracy": tp / max(tp + fp + fn, 1),
            "timing_err_ms_median": float(np.median(np.abs(errs))) if errs else np.nan,
            "timing_err_ms_iqr": float(np.subtract(*np.percentile(np.abs(errs), [75, 25])))
                                  if len(errs) > 3 else np.nan,
            "missed_rate": fn / max(len(gt), 1)}

def aggregate(rows):
    import pandas as pd
    df = pd.DataFrame(rows)
    out = {}
    for c in df.columns:
        if df[c].dtype.kind in "fi":
            out[c] = float(df[c].mean()); out[c + "_std"] = float(df[c].std())
    return out

def bland_altman(a, b):
    a, b = _f(a), _f(b)
    m = (a + b) / 2.0; d = a - b
    bias = float(d.mean()); sd = float(d.std())
    return {"mean": m, "diff": d, "bias": bias, "sd": sd,
            "loa_lo": bias - 1.96 * sd, "loa_hi": bias + 1.96 * sd}

def wilcoxon_holm(groups, better="higher"):
    # Pairwise Wilcoxon signed-rank across folds, Holm-corrected. groups: {name: [values]}
    import itertools
    names = list(groups)
    raw = []
    for a, b in itertools.combinations(names, 2):
        x, y = np.asarray(groups[a], float), np.asarray(groups[b], float)
        n = min(len(x), len(y))
        if n < 3 or np.allclose(x[:n], y[:n]):
            raw.append((a, b, np.nan)); continue
        try:
            p = float(sstats.wilcoxon(x[:n], y[:n]).pvalue)
        except Exception:
            p = np.nan
        raw.append((a, b, p))
    ps = [r[2] for r in raw]
    order = np.argsort([p if np.isfinite(p) else 1.0 for p in ps])
    m = len(ps); adj = [np.nan] * m; run = 0.0
    for k, i in enumerate(order):
        p = ps[i]
        if not np.isfinite(p):
            continue
        run = max(run, (m - k) * p)
        adj[i] = min(1.0, run)
    return [{"a": raw[i][0], "b": raw[i][1], "p": ps[i], "p_holm": adj[i]} for i in range(m)]
""",
 "crvs_models.py": r"""
# crvs_models.py -- the four baseline 1-D segmentation networks.
# All four are standardised the way Chowdhury et al. 2024 describe (section 3.1):
# 5 levels, 64 filters in the first level, doubling thereafter. Input (B, C_in, 1024).
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

def cbr(i, o, k=3, s=1):
    return nn.Sequential(nn.Conv1d(i, o, k, s, padding=k // 2, bias=False),
                         nn.BatchNorm1d(o), nn.ReLU(inplace=True))

class DoubleConv(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.b = nn.Sequential(cbr(i, o), cbr(o, o))
    def forward(self, x):
        return self.b(x)

# ------------------------------------------------------------------ UNet
class UNet1D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=64, levels=4):
        super().__init__()
        chs = [base * (2 ** i) for i in range(levels)]
        self.inc = DoubleConv(in_ch, chs[0])
        self.downs = nn.ModuleList()
        for i in range(levels - 1):
            self.downs.append(DoubleConv(chs[i], chs[i + 1]))
        self.bott = DoubleConv(chs[-1], chs[-1] * 2)
        self.ups = nn.ModuleList()
        self.decs = nn.ModuleList()
        prev = chs[-1] * 2
        for c in reversed(chs):
            self.ups.append(nn.ConvTranspose1d(prev, c, 4, 2, 1))
            self.decs.append(DoubleConv(c * 2, c))
            prev = c
        self.head = nn.Conv1d(chs[0], out_ch, 1)
    def forward(self, x):
        skips = []
        h = self.inc(x); skips.append(h)
        for d in self.downs:
            h = d(F.max_pool1d(h, 2)); skips.append(h)
        h = self.bott(F.max_pool1d(h, 2))
        for up, dec, sk in zip(self.ups, self.decs, reversed(skips)):
            h = up(h)
            if h.shape[-1] != sk.shape[-1]:
                h = F.interpolate(h, size=sk.shape[-1], mode="linear", align_corners=False)
            h = dec(torch.cat([h, sk], 1))
        return {"wave": torch.tanh(self.head(h))}

# ------------------------------------------------------------------ LinkNet
class LinkEnc(nn.Module):
    def __init__(self, i, o, stride=2):
        super().__init__()
        self.c1 = cbr(i, o, 3, stride)
        self.c2 = nn.Sequential(nn.Conv1d(o, o, 3, 1, 1, bias=False), nn.BatchNorm1d(o))
        self.sc = nn.Sequential(nn.Conv1d(i, o, 1, stride, bias=False), nn.BatchNorm1d(o))
    def forward(self, x):
        return F.relu(self.c2(self.c1(x)) + self.sc(x))

class LinkDec(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        m = max(i // 4, 8)
        self.a = cbr(i, m, 1)
        self.b = nn.Sequential(nn.ConvTranspose1d(m, m, 4, 2, 1, bias=False),
                               nn.BatchNorm1d(m), nn.ReLU(inplace=True))
        self.c = cbr(m, o, 1)
    def forward(self, x):
        return self.c(self.b(self.a(x)))

class LinkNet1D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=64, levels=4):
        super().__init__()
        chs = [base * (2 ** i) for i in range(levels)]
        self.stem = cbr(in_ch, chs[0], 7, 1)
        self.encs = nn.ModuleList()
        prev = chs[0]
        for c in chs:
            self.encs.append(LinkEnc(prev, c, 2)); prev = c
        self.bott = cbr(prev, prev)
        self.decs = nn.ModuleList()
        rev = list(reversed(chs))
        for k, c in enumerate(rev):
            nxt = rev[k + 1] if k + 1 < len(rev) else chs[0]
            self.decs.append(LinkDec(c, nxt))
        self.head = nn.Sequential(cbr(chs[0], chs[0]), nn.Conv1d(chs[0], out_ch, 1))
    def forward(self, x):
        h = self.stem(x)
        skips = []
        for e in self.encs:
            h = e(h); skips.append(h)
        h = self.bott(h)
        for k, d in enumerate(self.decs):
            h = d(h)
            j = len(skips) - 2 - k
            if j >= 0:
                s = skips[j]
                if h.shape[-1] != s.shape[-1]:
                    h = F.interpolate(h, size=s.shape[-1], mode="linear", align_corners=False)
                h = h + s
        return {"wave": torch.tanh(self.head(h))}

# ------------------------------------------------------------------ FPN
class FPN1D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=64, levels=4, pyr=128):
        super().__init__()
        chs = [base * (2 ** i) for i in range(levels)]
        self.stem = cbr(in_ch, chs[0], 7, 1)
        self.encs = nn.ModuleList()
        prev = chs[0]
        for c in chs:
            self.encs.append(LinkEnc(prev, c, 2)); prev = c
        self.lat = nn.ModuleList([nn.Conv1d(c, pyr, 1) for c in chs])
        self.smooth = nn.ModuleList([cbr(pyr, pyr) for _ in chs])
        self.heads = nn.ModuleList([nn.Sequential(cbr(pyr, pyr // 2), cbr(pyr // 2, pyr // 2))
                                    for _ in chs])
        self.head = nn.Sequential(cbr(pyr // 2, pyr // 2), nn.Conv1d(pyr // 2, out_ch, 1))
    def forward(self, x):
        L = x.shape[-1]
        h = self.stem(x); feats = []
        for e in self.encs:
            h = e(h); feats.append(h)
        ps = [None] * len(feats)
        ps[-1] = self.lat[-1](feats[-1])
        for i in range(len(feats) - 2, -1, -1):
            up = F.interpolate(ps[i + 1], size=feats[i].shape[-1], mode="linear",
                               align_corners=False)
            ps[i] = self.lat[i](feats[i]) + up
        ps = [s(p) for s, p in zip(self.smooth, ps)]
        acc = None
        for hd, p in zip(self.heads, ps):
            v = F.interpolate(hd(p), size=L, mode="linear", align_corners=False)
            acc = v if acc is None else acc + v
        return {"wave": torch.tanh(self.head(acc))}

# ------------------------------------------------------------------ MultiResLinkNet
class MultiResBlock(nn.Module):
    # MultiResUNet block (Ibtehaz & Rahman) in 1-D: three successive 3-conv stages of
    # increasing width, concatenated, plus a 1x1 residual shortcut.
    def __init__(self, cin, U, alpha=1.67):
        super().__init__()
        W = alpha * U
        # max(1, ...): below U=4 the 0.167 stage floors to zero channels, and the failure
        # then surfaces as an opaque conv error rather than pointing here.
        c1, c2, c3 = (max(1, int(W * 0.167)), max(1, int(W * 0.333)), max(1, int(W * 0.5)))
        self.out_channels = c1 + c2 + c3
        self.sc = nn.Sequential(nn.Conv1d(cin, self.out_channels, 1, bias=False),
                                nn.BatchNorm1d(self.out_channels))
        self.a = cbr(cin, c1); self.b = cbr(c1, c2); self.c = cbr(c2, c3)
        self.bn1 = nn.BatchNorm1d(self.out_channels)
        self.bn2 = nn.BatchNorm1d(self.out_channels)
    def forward(self, x):
        s = self.sc(x)
        a = self.a(x); b = self.b(a); c = self.c(b)
        o = self.bn1(torch.cat([a, b, c], 1))
        return F.relu(self.bn2(o + s))

class ResPath(nn.Module):
    # Processes an encoder feature before it is added to the decoder, instead of a raw skip.
    def __init__(self, ch, length):
        super().__init__()
        self.blocks = nn.ModuleList()
        for _ in range(max(1, length)):
            self.blocks.append(nn.ModuleDict({
                "sc": nn.Sequential(nn.Conv1d(ch, ch, 1, bias=False), nn.BatchNorm1d(ch)),
                "cv": nn.Sequential(nn.Conv1d(ch, ch, 3, padding=1, bias=False),
                                    nn.BatchNorm1d(ch)),
            }))
    def forward(self, x):
        for b in self.blocks:
            x = F.relu(b["sc"](x) + b["cv"](x))
        return x

class MultiResLinkNet1D(nn.Module):
    # LinkNet skeleton, MultiRes blocks instead of plain convolutions, ResPath skips added
    # (not concatenated), and deep supervision from every encoder level.
    def __init__(self, in_ch=1, out_ch=1, base=64, levels=4, deep_supervision=True):
        super().__init__()
        self.deep_supervision = deep_supervision
        units = [base * (2 ** i) for i in range(levels)]
        self.stem = cbr(in_ch, base, 7, 1)
        self.encs = nn.ModuleList(); self.paths = nn.ModuleList()
        prev = base; enc_ch = []
        for i, u in enumerate(units):
            blk = MultiResBlock(prev, u)
            self.encs.append(blk)
            self.paths.append(ResPath(blk.out_channels, levels - i))
            enc_ch.append(blk.out_channels); prev = blk.out_channels
        self.bott = MultiResBlock(prev, units[-1])
        self.decs = nn.ModuleList()
        rev_ch = list(reversed(enc_ch))
        cur = self.bott.out_channels
        # Decoder step k must emerge with the channel count AND length of skips[-1-k], or
        # the additive skip is silently dropped and every ResPath receives zero gradient.
        # Encoder here pools AFTER appending the skip, so the target is rev_ch[k] -- not
        # rev_ch[k+1], which is correct only for the stride-2 encoder in LinkNet1D.
        for k in range(levels):
            tgt = rev_ch[k]
            self.decs.append(LinkDec(cur, tgt)); cur = tgt
        self.head = nn.Sequential(cbr(cur, base), nn.Conv1d(base, out_ch, 1))
        self.aux = nn.ModuleList([nn.Conv1d(c, out_ch, 1) for c in enc_ch]) \
                   if deep_supervision else None
    def forward(self, x):
        L = x.shape[-1]
        h = self.stem(x)
        skips = []
        for e in self.encs:
            h = e(h); skips.append(h)
            h = F.max_pool1d(h, 2)
        h = self.bott(h)
        for k, d in enumerate(self.decs):
            h = d(h)
            j = len(skips) - 1 - k
            if j >= 0:
                s = self.paths[j](skips[j])
                if h.shape[-1] != s.shape[-1]:
                    h = F.interpolate(h, size=s.shape[-1], mode="linear", align_corners=False)
                if h.shape[1] != s.shape[1]:
                    raise RuntimeError(                     # raise, not assert: an invariant
                        f"skip channel mismatch at decoder {k}: {h.shape[1]} vs "        # this
                        f"{s.shape[1]}. Dropping it silently is what cost 59% of this "  # load
                        "model's gradient once already.")   # bearing must survive python -O
                h = h + s
        if h.shape[-1] != L:
            h = F.interpolate(h, size=L, mode="linear", align_corners=False)
        out = {"wave": torch.tanh(self.head(h))}
        if self.aux is not None and self.training:
            out["aux"] = [F.interpolate(a(s), size=L, mode="linear", align_corners=False)
                          for a, s in zip(self.aux, skips)]
        return out

BASELINES = {"fpn": FPN1D, "unet": UNet1D, "linknet": LinkNet1D,
             "multireslinknet": MultiResLinkNet1D}

def build_baseline(name, in_ch=1, out_ch=1, base=64, levels=4):
    return BASELINES[name](in_ch=in_ch, out_ch=out_ch, base=base, levels=levels)

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)
""",
 "crvs_losses.py": r"""
# crvs_losses.py -- C5, the morphology-aware composite loss.
# Plain MSE is the conditional mean, so it flattens the R peak; that is exactly why the
# baseline over-estimates RMSSD by ~2x. Every term here exists to stop that.
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiResSTFTLoss(nn.Module):
    # Spectral convergence + log-magnitude at three resolutions. Forces the model to get
    # the spectrum right, not just the sample-wise average.
    def __init__(self, ffts=(256, 128, 64)):
        super().__init__()
        self.ffts = ffts
    def _one(self, y, yh, n):
        hop, win = n // 4, n
        w = torch.hann_window(win, device=y.device, dtype=torch.float32)
        kw = dict(n_fft=n, hop_length=hop, win_length=win, window=w,
                  return_complex=True, center=True, pad_mode="reflect")
        Y = torch.stft(y, **kw).abs().clamp_min(1e-7)
        H = torch.stft(yh, **kw).abs().clamp_min(1e-7)
        sc = torch.norm(Y - H, p="fro", dim=(-2, -1)) / (torch.norm(Y, p="fro", dim=(-2, -1)) + 1e-7)
        mag = F.l1_loss(torch.log(H), torch.log(Y))
        return sc.mean() + mag
    def forward(self, y, yh):
        y = y.squeeze(1).float(); yh = yh.squeeze(1).float()
        return sum(self._one(y, yh, n) for n in self.ffts) / len(self.ffts)

def pearson_loss(y, yh, eps=1e-8):
    y = y.squeeze(1).float(); yh = yh.squeeze(1).float()
    y = y - y.mean(-1, keepdim=True); yh = yh - yh.mean(-1, keepdim=True)
    num = (y * yh).sum(-1)
    den = y.norm(dim=-1) * yh.norm(dim=-1) + eps
    return (1.0 - num / den).mean()

def focal_bce(logit, target, alpha=0.75, gamma=2.0):
    p = torch.sigmoid(logit)
    ce = F.binary_cross_entropy_with_logits(logit, target, reduction="none")
    pt = p * target + (1 - p) * (1 - target)
    w = alpha * target + (1 - alpha) * (1 - target)
    return (w * (1 - pt).pow(gamma) * ce).mean()

class CompositeLoss(nn.Module):
    def __init__(self, w_huber=1.0, w_stft=0.5, w_peak=0.3, w_rr=0.1,
                 w_peakw=0.5, w_corr=0.3, huber_delta=0.1, peak_weight=4.0):
        super().__init__()
        self.w = dict(huber=w_huber, stft=w_stft, peak=w_peak, rr=w_rr,
                      peakw=w_peakw, corr=w_corr)
        self.delta = huber_delta
        self.peak_weight = peak_weight
        self.stft = MultiResSTFTLoss()
    def forward(self, pred, y, pk=None, rr=None):
        parts = {}
        wave = pred["wave"]
        if self.w["huber"]:
            parts["huber"] = F.huber_loss(wave, y, delta=self.delta)
        if self.w["stft"]:
            parts["stft"] = self.stft(y, wave)
        if self.w["corr"]:
            parts["corr"] = pearson_loss(y, wave)
        if self.w["peakw"] and pk is not None:
            wgt = 1.0 + self.peak_weight * pk
            parts["peakw"] = ((wgt * (wave - y).abs()).sum() / (wgt.sum() + 1e-8))
        if self.w["peak"] and pk is not None and "peak" in pred:
            parts["peak"] = focal_bce(pred["peak"], pk)
        if self.w["rr"] and rr is not None and "rr" in pred:
            parts["rr"] = F.l1_loss(pred["rr"], rr)
        if "aux" in pred:
            parts["aux"] = sum(F.huber_loss(a, y, delta=self.delta)
                               for a in pred["aux"]) / max(len(pred["aux"]), 1) * 0.2
        total = sum(self.w.get(k, 1.0) * v for k, v in parts.items())
        return total, {k: float(v.detach()) for k, v in parts.items()}

class MSEOnly(nn.Module):
    # The baseline's objective, kept verbatim so ablation row 1 is a true reproduction.
    def forward(self, pred, y, pk=None, rr=None):
        l = F.mse_loss(pred["wave"], y)
        if "aux" in pred:
            l = l + 0.2 * sum(F.mse_loss(a, y) for a in pred["aux"]) / max(len(pred["aux"]), 1)
        return l, {"mse": float(l.detach())}
""",
 "crvs_engine.py": r"""
# crvs_engine.py -- the shared GPU training engine.
# Dual T4 via DataParallel, AMP, cosine schedule, early stopping, and a checkpoint written
# EVERY epoch so a killed session costs nothing. Runs are queued and skipped if already done.
import json, math, time, os
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

def _autocast(device_type, enabled):
    # torch.cuda.amp.autocast / GradScaler are deprecated and warn on every step in
    # torch >= 2.4. Use the device-typed API where it exists, fall back where it does not.
    try:
        return torch.amp.autocast(device_type=device_type, enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.autocast(enabled=enabled)

def _grad_scaler(device_type, enabled):
    try:
        return torch.amp.GradScaler(device_type, enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)

def pick_device():
    if torch.cuda.is_available():
        n = torch.cuda.device_count()
        names = [torch.cuda.get_device_name(i) for i in range(n)]
        return torch.device("cuda"), n, names
    return torch.device("cpu"), 0, []

def seed_all(s):
    import random
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

class Trainer:
    def __init__(self, model, loss_fn, out_dir, run_id, sync=None, lr=5e-4, weight_decay=1e-4,
                 epochs=120, patience=20, batch_size=64, num_workers=2, amp=True,
                 multi_gpu=True, grad_clip=1.0, min_lr=1e-6, log_every=50):
        self.device, self.ngpu, self.gpu_names = pick_device()
        self.raw_model = model.to(self.device)
        self.model = self.raw_model
        if multi_gpu and self.ngpu > 1:
            self.model = nn.DataParallel(self.raw_model)
        self.loss_fn = loss_fn
        self.out = Path(out_dir); self.out.mkdir(parents=True, exist_ok=True)
        self.run_id = run_id; self.sync = sync
        self.epochs = epochs; self.patience = patience
        self.bs = batch_size; self.nw = num_workers
        self.amp = amp and self.device.type == "cuda"
        self.grad_clip = grad_clip
        self.opt = torch.optim.AdamW(self.raw_model.parameters(), lr=lr,
                                     weight_decay=weight_decay)
        self.sched = torch.optim.lr_scheduler.CosineAnnealingLR(self.opt, T_max=epochs,
                                                                eta_min=min_lr)
        self.scaler = _grad_scaler(self.device.type, self.amp)
        self.log_every = log_every
        self.state = {"epoch": 0, "best": float("inf"), "best_epoch": -1, "history": [],
                      "run_id": run_id, "done": False}

    @property
    def ckpt(self):
        return self.out / "state.pt"

    def save(self, tag="state"):
        torch.save({"model": self.raw_model.state_dict(),
                    "opt": self.opt.state_dict(),
                    "sched": self.sched.state_dict(),
                    "scaler": self.scaler.state_dict(),
                    "state": self.state,
                    "torch_rng": torch.get_rng_state(),
                    "np_rng": np.random.get_state()},
                   self.out / (tag + ".pt"))
        (self.out / "state.json").write_text(json.dumps(self.state, indent=2, default=str))

    def load(self):
        if not self.ckpt.exists():
            return False
        try:
            d = torch.load(self.ckpt, map_location=self.device, weights_only=False)
            self.raw_model.load_state_dict(d["model"])
            self.opt.load_state_dict(d["opt"]); self.sched.load_state_dict(d["sched"])
            self.scaler.load_state_dict(d["scaler"]); self.state = d["state"]
            try:
                torch.set_rng_state(d["torch_rng"].cpu()); np.random.set_state(d["np_rng"])
            except Exception:
                pass
            print(f"  resumed {self.run_id} at epoch {self.state['epoch']}")
            return True
        except Exception as e:
            print(f"  checkpoint unreadable ({type(e).__name__}), starting fresh")
            return False

    def _loader(self, ds, shuffle):
        if len(ds) == 0:
            raise RuntimeError("empty dataset -- check the fold split; training on nothing "
                               "would produce a plausible-looking but untrained checkpoint")
        # drop_last=True on a split smaller than one batch yields ZERO batches, the optimiser
        # never steps, and the loss is reported as 0.00000. Guard it explicitly.
        drop = bool(shuffle) and len(ds) > self.bs
        if shuffle and not drop:
            print(f"  note: only {len(ds)} train window(s) < batch {self.bs}; keeping the "
                  f"partial batch so the optimiser actually steps")
        return DataLoader(ds, batch_size=min(self.bs, max(len(ds), 1)), shuffle=shuffle,
                          num_workers=self.nw, pin_memory=(self.device.type == "cuda"),
                          drop_last=drop, persistent_workers=self.nw > 0)

    def _step(self, batch, train):
        x, y, pk, rr = [b.to(self.device, non_blocking=True) for b in batch]
        with _autocast(self.device.type, self.amp):
            pred = self.model(x)
            if isinstance(pred, dict) and "aux" in pred and not train:
                pred = {k: v for k, v in pred.items() if k != "aux"}
            loss, parts = self.loss_fn(pred, y, pk, rr)
        return loss, parts, pred, y

    def fit(self, train_ds, val_ds):
        tl = self._loader(train_ds, True)
        vl = self._loader(val_ds, False)
        start = self.state["epoch"]
        # Only the epoch counter decides completion. Keying off a sticky `done` flag meant
        # raising CFG["EPOCHS"] later silently no-opped instead of training further.
        if start >= self.epochs:
            print(f"  {self.run_id} already complete at epoch {start}/{self.epochs}")
            return self.state
        if self.state.get("done"):
            print(f"  extending {self.run_id}: {start} -> {self.epochs} epochs")
            self.state["done"] = False
        bad = 0
        for ep in range(start, self.epochs):
            self.model.train(); t0 = time.time(); tot = 0.0; n = 0
            for i, batch in enumerate(tl):
                self.opt.zero_grad(set_to_none=True)
                loss, parts, _, _ = self._step(batch, True)
                self.scaler.scale(loss).backward()
                if self.grad_clip:
                    self.scaler.unscale_(self.opt)
                    torch.nn.utils.clip_grad_norm_(self.raw_model.parameters(), self.grad_clip)
                self.scaler.step(self.opt); self.scaler.update()
                tot += float(loss.detach()); n += 1
            if n == 0:
                raise RuntimeError(
                    "the training loader yielded zero batches -- the optimiser never stepped. "
                    "This would write a checkpoint that looks trained and is not.")
            self.sched.step()
            tr = tot / n
            self.model.eval(); vtot = 0.0; vn = 0
            with torch.no_grad():
                for batch in vl:
                    loss, _, _, _ = self._step(batch, False)
                    vtot += float(loss); vn += 1
            va = vtot / max(vn, 1)
            rec = {"epoch": ep + 1, "train": tr, "val": va,
                   "lr": self.opt.param_groups[0]["lr"], "sec": round(time.time() - t0, 1)}
            self.state["history"].append(rec); self.state["epoch"] = ep + 1
            improved = va < self.state["best"] - 1e-6
            if improved:
                self.state["best"] = va; self.state["best_epoch"] = ep + 1; bad = 0
                self.save("best")
            else:
                bad += 1
            self.save("state")
            if self.sync:
                self.sync.log("epoch", run=self.run_id, **rec, best=round(self.state["best"], 6))
                if improved:
                    self.sync.stage_done(f"{self.run_id}:best@{ep+1}")
            print(f"  ep {ep+1:>3}/{self.epochs}  train {tr:.5f}  val {va:.5f}"
                  f"{'  *' if improved else ''}  {rec['sec']:.0f}s")
            if bad >= self.patience:
                print(f"  early stop at epoch {ep+1} (no improvement for {self.patience})")
                break
        self.state["done"] = True; self.save("state")
        if self.sync:
            self.sync.stage_done(f"{self.run_id}:done")
        return self.state

    @torch.no_grad()
    def predict(self, ds, max_keep=200):
        bp = self.out / "best.pt"
        if bp.exists():
            try:
                self.raw_model.load_state_dict(
                    torch.load(bp, map_location=self.device, weights_only=False)["model"])
            except Exception as e:
                print("  could not load best.pt:", e)
        self.model.eval()
        dl = self._loader(ds, False)
        Y, P = [], []
        for batch in dl:
            x, y, pk, rr = [b.to(self.device, non_blocking=True) for b in batch]
            with _autocast(self.device.type, self.amp):
                out = self.model(x)
            Y.append(y.squeeze(1).float().cpu().numpy())
            P.append(out["wave"].squeeze(1).float().cpu().numpy())
        Y = np.concatenate(Y, 0); P = np.concatenate(P, 0)
        return Y, P
""",
}
for nm, src in MODULES.items():
    (WORK / nm).write_text(src)
    print(f"  {nm:<20} {len(src):>7,} chars")

# Writing a .py and importing it is NOT idempotent inside one kernel: Python caches the
# module object in sys.modules, so re-running this cell after updating the notebook keeps
# the OLD code. That is exactly how a stale .npz loader survived a rebuilt notebook and
# produced a FileNotFoundError deep inside a DataLoader worker. Purge and re-import.
import importlib
for nm in MODULES:
    sys.modules.pop(nm[:-3], None)
importlib.invalidate_caches()

import crvs_data
REQUIRED_LIB = 3
if getattr(crvs_data, "LIB_VERSION", 0) < REQUIRED_LIB:
    raise RuntimeError(
        f"\n{'='*74}\n  Stale crvs_data: version "
        f"{getattr(crvs_data, 'LIB_VERSION', 'missing')}, need >= {REQUIRED_LIB}."
        f"\n  Loaded from {getattr(crvs_data, '__file__', '?')}"
        f"\n  Restart the kernel (Run -> Restart & Run All) and try again.\n{'='*74}")
print(f"\nlibrary written to {WORK}  |  crvs_data v{crvs_data.LIB_VERSION} loaded from "
      f"{crvs_data.__file__}")

In [ ]:
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if not HF_TOKEN:
        raise RuntimeError("\n" + "="*74 +
            "\n  HF_TOKEN not found. Add-ons -> Secrets -> HF_TOKEN (write) -> attach.\n" + "="*74)

from crvs_sync import HFSync
sync = HFSync(repo_id=CFG["DST_REPO"], local_dir=WORK, token=HF_TOKEN, repo_type="model",
              private=CFG["HF_PRIVATE"], run_id=CFG["RUN_ID"],
              push_interval_s=CFG["PUSH_INTERVAL_S"], max_req_hour=CFG["HF_MAX_REQ_HOUR"])
print("\nresults repo:", sync.url, "(public)")

_M = {"f": False, "n": ""}
def MAJOR(nm):
    _M["f"] = True; _M["n"] = nm
def _hook(r=None):
    if _M["f"]:
        nm = _M["n"]; _M["f"] = False; _M["n"] = ""; sync.stage_done(nm)
try:
    get_ipython().events.register("post_run_cell", _hook)
    print("post-run-cell push hook registered")
except Exception as e:
    print("hook unavailable:", e)

# Resume: pull back the small artefacts (state, summaries, metrics) but NOT the weights --
# finished runs never need their checkpoints re-downloaded, only their summary.json.
sync.pull(allow_patterns=["*.json", "*.jsonl", "*.csv", "*.md", "runs/**/summary.json",
                          "runs/**/state.json", "results/*"])
STATE = sync.load_state({"completed": [], "sessions": 0})
STATE["sessions"] = STATE.get("sessions", 0) + 1
sync.save_state(STATE)
print(f"session #{STATE['sessions']}  |  {len(STATE['completed'])} run(s) already complete")
MAJOR("00_setup")

---
# 3 · Fetch the corpus

Recordings are pulled to **scratch**, not to `/kaggle/working`, so the ~400 MB of data never eats
into the 20 GB output budget that the checkpoints need.

In [ ]:
from huggingface_hub import snapshot_download
DATA = SCRATCH / "corpus"
t0 = time.time()
snapshot_download(CFG["SRC_REPO"], repo_type="dataset", token=HF_TOKEN, local_dir=str(DATA),
                  allow_patterns=["recordings/*.npy", "recordings/*.json",
                                  "recordings/*.npz",   # legacy corpus still works
                                  "windows.parquet", "recordings.csv",
                                  "norm_stats.json", "experiments.json"])
print(f"corpus downloaded in {time.time()-t0:.0f}s")

W = pd.read_parquet(DATA / "windows.parquet")
RECS = pd.read_csv(DATA / "recordings.csv")
NORM = json.loads((DATA / "norm_stats.json").read_text())
EXPINFO = json.loads((DATA / "experiments.json").read_text())
EXPERIMENTS = EXPINFO["experiments"]
REC_DIR = DATA / "recordings"

# Fail here, loudly, rather than inside a DataLoader worker 20 minutes into a run.
_npy = {p.stem for p in (REC_DIR).glob("*.npy")}
_npz = {p.stem for p in (REC_DIR).glob("*.npz")}
_have = _npy | _npz
print(f"recording files: {len(_npy)} .npy (fast path) + {len(_npz)} .npz (legacy, ~85x "
      f"slower per window)")
if not _have:
    raise RuntimeError(
        "\n" + "=" * 74 +
        "\n  No recording files downloaded."
        "\n  Check that NB02 finished and pushed, and that SRC_REPO matches its DST_REPO."
        "\n" + "=" * 74)
_missing = sorted(set(RECS["rec_id"]) - _have)
if _missing:
    print(f"WARNING: {len(_missing)} recording(s) in recordings.csv have no file: "
          f"{_missing[:5]}{' ...' if len(_missing) > 5 else ''}")
    W = W[~W["rec_id"].isin(_missing)].reset_index(drop=True)
    RECS = RECS[~RECS["rec_id"].isin(_missing)].reset_index(drop=True)
    print(f"         dropped their windows; {len(W):,} remain")

print(f"windows      : {len(W):,}   ({int(W['no_overlap'].sum()):,} non-overlapping)")
print(f"recordings   : {len(RECS)}   subjects: {RECS['subject'].nunique()}")
print(f"norm sets    : {len(NORM)}")
print(f"channels     : {EXPINFO['channels']}")
print(f"using        : {CFG['CHANNELS']}  <- the baseline's single-channel input")
print("\nwindows per experiment:")
for e, sc in EXPERIMENTS.items():
    sub = W[W["scenario_canon"].isin(sc)]
    print(f"  {e:<12} {len(sub):>8,} windows  {sub['subject'].nunique():>3} subjects  {sc}")

---
# 4 · Model smoke test

Before spending GPU hours, prove each network actually runs: correct output shape, finite values,
gradients that flow, and a parameter count we can report. The baseline paper reports **no**
parameter or FLOP budget for any of its models — we will, for all of them.

In [ ]:
from crvs_models import build_baseline, count_params
from crvs_losses import MSEOnly, CompositeLoss
from crvs_engine import Trainer, seed_all, pick_device

seed_all(CFG["SEED"])
dev, ngpu, names = pick_device()
print(f"device: {dev}  gpus: {ngpu} {names}\n")

C_IN = len(CFG["CHANNELS"])
x = torch.randn(4, C_IN, 1024, device=dev)
y = torch.randn(4, 1, 1024, device=dev).clamp(-1, 1)
rows = []
print(f"{'model':<20}{'params':>12}{'MB':>8}{'out shape':>18}{'fwd ms':>9}  grad")
print("-" * 78)
for nm in CFG["MODELS"]:
    m = build_baseline(nm, in_ch=C_IN, out_ch=1, base=CFG["BASE"], levels=CFG["LEVELS"]).to(dev)
    m.train()
    t0 = time.time()
    out = m(x)
    if dev.type == "cuda":
        torch.cuda.synchronize()
    dt = (time.time() - t0) * 1000
    loss = torch.nn.functional.mse_loss(out["wave"], y)
    if "aux" in out:
        loss = loss + sum(torch.nn.functional.mse_loss(a, y) for a in out["aux"]) * 0.1
    loss.backward()
    gn = sum(float(p.grad.norm()) for p in m.parameters() if p.grad is not None)
    p = count_params(m)
    ok = (out["wave"].shape == y.shape and torch.isfinite(out["wave"]).all() and gn > 0)
    rows.append({"model": nm, "params": p, "mb": p * 4 / 2**20, "ok": bool(ok)})
    print(f"{nm:<20}{p:>12,}{p*4/2**20:>8.1f}{str(tuple(out['wave'].shape)):>18}"
          f"{dt:>9.1f}  {'OK' if ok else 'FAIL'}")
    del m, out, loss
    gc.collect()
    if dev.type == "cuda":
        torch.cuda.empty_cache()
assert all(r["ok"] for r in rows), "a baseline failed its smoke test"
pd.DataFrame(rows).to_csv(WORK / "results" / "model_budget.csv", index=False)
print("\nall four baselines forward, backward and produce finite output.")
MAJOR("01_smoke")

---
# 5 · The run queue

Each entry is one (experiment, model, fold). Completed runs are read from HF state and skipped,
so re-running the notebook in a fresh session simply continues.

`QUICK = True` collapses this to one fold and 25 epochs — enough to confirm the whole path works
and to see whether the losses look sane, without committing a full session.

In [ ]:
folds = range(CFG["QUICK_FOLDS"] if CFG["QUICK"] else CFG["N_FOLDS"])
EPOCHS = CFG["QUICK_EPOCHS"] if CFG["QUICK"] else CFG["EPOCHS"]

QUEUE = []
for exp in CFG["EXPERIMENTS"]:
    for mdl in CFG["MODELS"]:
        for f in folds:
            QUEUE.append({"run_id": f"{exp}__{mdl}__f{f}", "exp": exp, "model": mdl, "fold": f})

done = set(STATE.get("completed", []))
todo = [q for q in QUEUE if q["run_id"] not in done]
print(f"queue: {len(QUEUE)} run(s) total | {len(done)} done | {len(todo)} remaining")
print(f"epochs per run: {EPOCHS}   time budget: {CFG['TIME_BUDGET_H']} h")
if CFG["QUICK"]:
    print("\n>>> QUICK MODE. Confirm this completes, then set CFG['QUICK']=False and re-run.")
print("\nnext up:")
for q in todo[:8]:
    print("   ", q["run_id"])
if len(todo) > 8:
    print(f"    ... and {len(todo)-8} more")

In [ ]:
from crvs_data import WindowDataset, WINDOW, FS
from crvs_metrics import seg_metrics, detect_r_peaks, hrv_from_peaks, peak_detection_scores

def split_for(exp, fold, n_folds=None):
    n_folds = n_folds or CFG["N_FOLDS"]
    sub = W[W["scenario_canon"].isin(EXPERIMENTS[exp])]
    te_g, va_g = fold % n_folds, (fold + 1) % n_folds
    tr = sub[~sub["fold_group"].isin([te_g, va_g])]
    va = sub[(sub["fold_group"] == va_g) & sub["no_overlap"]]
    te = sub[(sub["fold_group"] == te_g) & sub["no_overlap"]]
    assert not (set(tr["subject"]) & set(te["subject"])), "SUBJECT LEAK"
    return tr, va, te

def make_datasets(exp, fold):
    tr, va, te = split_for(exp, fold)
    norm = NORM.get(f"{exp}|{fold}")
    if norm is None:
        raise RuntimeError(f"no normalisation stats for {exp}|{fold} -- re-run NB02")
    idx = [EXPINFO["channels"].index(c) for c in CFG["CHANNELS"]]
    sub_norm = {"mean": [norm["mean"][i] for i in idx],
                "std":  [norm["std"][i] for i in idx]}
    mk = lambda d, aug: WindowDataset(REC_DIR, d, sub_norm, CFG["CHANNELS"], augment=aug,
                                      seed=CFG["SEED"] + fold)
    return mk(tr, True), mk(va, False), mk(te, False), (tr, va, te)

def evaluate(Y, P, index, out_dir):
    # Per-window metrics, then per-subject and overall aggregates. Saving per-window rows
    # lets NB05 run the statistics without ever re-running a model.
    rows = []
    subs = index["subject"].to_numpy()
    scen = index["scenario_canon"].to_numpy()
    for i in range(len(Y)):
        m = seg_metrics(Y[i], P[i], FS)
        m["subject"] = subs[i] if i < len(subs) else "?"
        m["scenario"] = scen[i] if i < len(scen) else "?"
        rows.append(m)
    dfw = pd.DataFrame(rows)
    dfw.to_parquet(out_dir / "metrics_windows.parquet", index=False)
    num = [c for c in dfw.columns if dfw[c].dtype.kind in "fi"]
    agg = {c: float(dfw[c].mean()) for c in num}
    agg.update({c + "_std": float(dfw[c].std()) for c in num})
    # continuous-signal HR / HRV, computed on the concatenated test signal per subject
    hr_rows = []
    for s in pd.unique(subs):
        sel = subs == s
        if sel.sum() < 2:
            continue
        yg = np.concatenate(Y[sel]); yp = np.concatenate(P[sel])
        g = hrv_from_peaks(detect_r_peaks(yg, FS), FS)
        p = hrv_from_peaks(detect_r_peaks(yp, FS), FS)
        pk = peak_detection_scores(yg, yp, FS)
        hr_rows.append({"subject": s, **{f"gt_{k}": v for k, v in g.items()},
                        **{f"pr_{k}": v for k, v in p.items()}, **pk})
    dfh = pd.DataFrame(hr_rows)
    if len(dfh):
        dfh.to_parquet(out_dir / "metrics_subjects.parquet", index=False)
        for k in ("F1", "precision", "recall", "accuracy", "missed_rate",
                  "timing_err_ms_median"):
            if k in dfh.columns:
                agg["peak_" + k] = float(dfh[k].mean())
        for k in ("mean_hr_bpm", "rmssd_ms"):
            if f"gt_{k}" in dfh and f"pr_{k}" in dfh:
                agg["MAE_" + k] = float((dfh[f"gt_{k}"] - dfh[f"pr_{k}"]).abs().mean())
    return agg, dfw

---
# 6 · Train

The loop below is the whole notebook. For each queued run it builds the split, trains with early
stopping, evaluates on the held-out subjects, writes everything under `runs/<run_id>/`, marks the
run complete and pushes.

Watch the `val` column. If it stops improving in the first few epochs across every model, the
targets are probably misaligned — stop and check NB02's `nb02_fig1_window.png`.

**You can interrupt at any point.** The current epoch's checkpoint is already on disk, the
interrupt handler pushes it, and the next session resumes mid-run.

In [ ]:
t_start = time.time()
budget = CFG["TIME_BUDGET_H"] * 3600
completed_now = []

for qi, q in enumerate(todo, 1):
    el = time.time() - t_start
    if el > budget:
        print(f"\n=== time budget reached ({el/3600:.2f} h). Stopping cleanly. ===")
        print(f"    {len(todo)-qi+1} run(s) left -- start a new session and re-run this notebook.")
        break
    rid, exp, mdl, fold = q["run_id"], q["exp"], q["model"], q["fold"]
    out = WORK / "runs" / rid
    out.mkdir(parents=True, exist_ok=True)
    print("\n" + "=" * 78)
    print(f"[{qi}/{len(todo)}]  {rid}   ({el/3600:.2f} h elapsed)")
    print("=" * 78)
    try:
        tr_ds, va_ds, te_ds, (tri, vai, tei) = make_datasets(exp, fold)
        print(f"  train {len(tr_ds):,} | val {len(va_ds):,} | test {len(te_ds):,} windows   "
              f"test subjects: {sorted(tei['subject'].unique())}")
        seed_all(CFG["SEED"] + fold)
        model = build_baseline(mdl, in_ch=len(CFG["CHANNELS"]), out_ch=1,
                               base=CFG["BASE"], levels=CFG["LEVELS"])
        loss_fn = MSEOnly() if CFG["LOSS"] == "mse" else CompositeLoss()
        tr = Trainer(model, loss_fn, out, rid, sync=sync, lr=CFG["LR"],
                     weight_decay=CFG["WEIGHT_DECAY"], epochs=EPOCHS,
                     patience=CFG["PATIENCE"], batch_size=CFG["BATCH"],
                     num_workers=CFG["WORKERS"], amp=CFG["AMP"],
                     multi_gpu=CFG["MULTI_GPU"])
        tr.load()
        tr.fit(tr_ds, va_ds)
        Y, P = tr.predict(te_ds)
        agg, dfw = evaluate(Y, P, tei, out)

        keep = min(200, len(Y))
        sel = np.linspace(0, len(Y) - 1, keep).astype(int)
        np.savez_compressed(out / "preds_sample.npz", y=Y[sel].astype(np.float32),
                            p=P[sel].astype(np.float32),
                            subject=tei["subject"].to_numpy()[sel].astype(str))
        summary = {"run_id": rid, "experiment": exp, "model": mdl, "fold": fold,
                   "channels": CFG["CHANNELS"], "loss": CFG["LOSS"], "epochs_run": tr.state["epoch"],
                   "best_epoch": tr.state["best_epoch"], "best_val": tr.state["best"],
                   "params": count_params(tr.raw_model),
                   "n_train": len(tr_ds), "n_val": len(va_ds), "n_test": len(te_ds),
                   "test_subjects": sorted(map(str, tei["subject"].unique())),
                   "metrics": agg,
                   "finished_utc": datetime.now(timezone.utc).isoformat()}
        (out / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
        print(f"  --> CC_t {agg['CC_temporal']:.2f}  CC_s {agg['CC_spectral']:.2f}  "
              f"MAE {agg['MAE']:.5f}  MSE {agg['MSE']:.5f}  "
              f"RRMSE_t {agg['RRMSE_temporal']:.4f}  F1 {agg.get('peak_F1', float('nan')):.3f}")
        done.add(rid); completed_now.append(rid)
        STATE["completed"] = sorted(done); sync.save_state(STATE)
        sync.stage_done(f"run:{rid}", cc_t=round(agg["CC_temporal"], 2))
        del tr, model, tr_ds, va_ds, te_ds, Y, P
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except KeyboardInterrupt:
        print("\ninterrupted -- checkpoint saved and pushed; re-run to resume this exact run.")
        raise
    except Exception as e:
        import traceback
        print(f"  !! {type(e).__name__}: {e}")
        (out / "error.txt").write_text(traceback.format_exc())
        sync.log("run_failed", run=rid, err=f"{type(e).__name__}: {e}")

print(f"\ncompleted this session: {len(completed_now)}   total: {len(done)}/{len(QUEUE)}")
MAJOR("02_training")

---
# 7 · Results, next to the published table

The comparison that decides whether we may proceed. Our MultiResLinkNet row on `B_rva` is the one
that matters: the baseline reports **CC_temporal 61.86, CC_spectral 79.96, MAE 0.14841,
RRMSE_t 0.44618**.

Read the gap in the right direction. Because our splits are subject-wise with non-overlapping test
windows, landing **somewhat below** their figures is the expected and honest outcome — it means we
removed the leakage, not that our implementation is worse. What must hold is the **ordering**:
MultiResLinkNet should still beat FPN, UNet and LinkNet on correlation. If the ordering inverts,
the reimplementation is wrong.

In [ ]:
rows = []
for p in sorted((WORK / "runs").glob("*/summary.json")):
    try:
        s = json.loads(p.read_text())
        rows.append({"experiment": s["experiment"], "model": s["model"], "fold": s["fold"],
                     "params": s.get("params"), "best_epoch": s.get("best_epoch"),
                     **{k: v for k, v in s["metrics"].items() if not k.endswith("_std")}})
    except Exception:
        pass
R = pd.DataFrame(rows)
if not len(R):
    print("no completed runs yet -- run the training cell.")
else:
    R.to_csv(WORK / "results" / "runs_raw.csv", index=False)
    cols = ["MAE", "MSE", "CC_temporal", "CC_spectral", "RRMSE_temporal", "RRMSE_spectral"]
    agg = (R.groupby(["experiment", "model"])[cols + ["params"]]
             .agg(["mean", "std"]).round(5))
    print("=" * 100); print("OUR RUNS  (mean +/- std across folds)"); print("=" * 100)
    flat = R.groupby(["experiment", "model"])[cols].mean().round(5).reset_index()
    nfold = R.groupby(["experiment", "model"]).size().rename("folds").reset_index()
    flat = flat.merge(nfold, on=["experiment", "model"])
    print(flat.to_string(index=False))
    flat.to_csv(WORK / "results" / "runs_summary.csv", index=False)

    PAPER = {
      ("B_rva","fpn"):            dict(MAE=.14316, MSE=.03422, CC_temporal=59.63, CC_spectral=69.53, RRMSE_temporal=.44694, RRMSE_spectral=.83026),
      ("B_rva","unet"):           dict(MAE=.14798, MSE=.03741, CC_temporal=57.65, CC_spectral=68.39, RRMSE_temporal=.45315, RRMSE_spectral=.94118),
      ("B_rva","linknet"):        dict(MAE=.14780, MSE=.03723, CC_temporal=58.69, CC_spectral=70.91, RRMSE_temporal=.45487, RRMSE_spectral=.86909),
      ("B_rva","multireslinknet"):dict(MAE=.14841, MSE=.03793, CC_temporal=61.86, CC_spectral=79.96, RRMSE_temporal=.44618, RRMSE_spectral=.73269),
      ("A_resting","fpn"):            dict(CC_temporal=58.37, CC_spectral=71.38),
      ("A_resting","unet"):           dict(CC_temporal=63.10, CC_spectral=74.68),
      ("A_resting","linknet"):        dict(CC_temporal=64.35, CC_spectral=74.37),
      ("A_resting","multireslinknet"):dict(CC_temporal=66.10, CC_spectral=82.44),
      ("A_valsalva","fpn"):            dict(CC_temporal=57.53, CC_spectral=65.97),
      ("A_valsalva","unet"):           dict(CC_temporal=58.38, CC_spectral=68.79),
      ("A_valsalva","linknet"):        dict(CC_temporal=56.63, CC_spectral=66.87),
      ("A_valsalva","multireslinknet"):dict(CC_temporal=60.14, CC_spectral=77.05),
      ("A_apnea","fpn"):            dict(CC_temporal=39.12, CC_spectral=51.26),
      ("A_apnea","unet"):           dict(CC_temporal=56.14, CC_spectral=69.97),
      ("A_apnea","linknet"):        dict(CC_temporal=56.22, CC_spectral=70.35),
      ("A_apnea","multireslinknet"):dict(CC_temporal=55.33, CC_spectral=74.66),
    }
    cmp_rows = []
    for _, r in flat.iterrows():
        p = PAPER.get((r["experiment"], r["model"]), {})
        cmp_rows.append({"experiment": r["experiment"], "model": r["model"], "folds": r["folds"],
                         "CC_t_ours": round(r["CC_temporal"], 2),
                         "CC_t_paper": p.get("CC_temporal"),
                         "CC_t_delta": (round(r["CC_temporal"] - p["CC_temporal"], 2)
                                        if "CC_temporal" in p else None),
                         "CC_s_ours": round(r["CC_spectral"], 2),
                         "CC_s_paper": p.get("CC_spectral"),
                         "MAE_ours": round(r["MAE"], 5), "MAE_paper": p.get("MAE")})
    CMP = pd.DataFrame(cmp_rows)
    print("\n" + "=" * 100); print("OURS vs. PUBLISHED"); print("=" * 100)
    print(CMP.to_string(index=False))
    CMP.to_csv(WORK / "results" / "vs_paper.csv", index=False)

    b = flat[flat["experiment"] == "B_rva"].sort_values("CC_temporal", ascending=False)
    if len(b):
        print("\n" + "-" * 100)
        print("REPRODUCTION GATE  (Experiment B, RVA combined)")
        print("-" * 100)
        print(f"  ranking by CC_temporal: {' > '.join(b['model'].tolist())}")
        top = b.iloc[0]["model"]
        ok = top == "multireslinknet"
        print(f"  MultiResLinkNet ranks first: {ok}")
        mr = b[b["model"] == "multireslinknet"]
        if len(mr):
            v = float(mr.iloc[0]["CC_temporal"])
            print(f"  our CC_temporal {v:.2f}  vs published 61.86  (delta {v-61.86:+.2f})")
        print("\n  " + ("GATE PASSED -- ordering reproduced, proceed to NB04."
                        if ok else
                        "GATE NOT PASSED -- MultiResLinkNet is not top. Check the reimplementation"
                        "\n  before trusting anything downstream. (With QUICK=True and one fold this"
                        "\n  is common; re-check with the full queue.)"))
MAJOR("03_results")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
S = {"radar": "#0F7C82", "ecg": "#AF3A2C", "muted": "#5C6B71", "ink": "#10171B",
     "grid": "#D3DADB", "amber": "#8A6212"}
plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 160, "savefig.bbox": "tight",
                     "axes.grid": True, "grid.color": S["grid"], "grid.linewidth": .6,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 8.5, "axes.titlesize": 10, "axes.titleweight": "bold"})
FIG = WORK / "figures"

if len(R):
    b = R[R["experiment"] == "B_rva"]
    if len(b):
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
        order = ["fpn", "unet", "linknet", "multireslinknet"]
        order = [o for o in order if o in set(b["model"])]
        for ax, met, ref in [(axes[0], "CC_temporal",
                              {"fpn":59.63,"unet":57.65,"linknet":58.69,"multireslinknet":61.86}),
                             (axes[1], "CC_spectral",
                              {"fpn":69.53,"unet":68.39,"linknet":70.91,"multireslinknet":79.96})]:
            vals = [b[b["model"] == o][met].mean() for o in order]
            ax.bar(order, vals, color=[S["radar"]] * (len(order) - 1) + [S["ecg"]],
                   edgecolor="white", label="ours")
            ax.plot(order, [ref[o] for o in order], "o--", color=S["ink"], ms=5, lw=1.2,
                    label="published")
            ax.set_title(met + "   (Experiment B, RVA)")
            ax.tick_params(axis="x", rotation=18)
            ax.legend(frameon=False, fontsize=7)
        fig.tight_layout(); fig.savefig(FIG / "nb03_fig1_vs_paper.png"); plt.close(fig)
        print("  wrote nb03_fig1_vs_paper.png")

    fig, ax = plt.subplots(figsize=(9, 3.2))
    for m in sorted(set(R["model"])):
        hs = []
        for p in (WORK / "runs").glob(f"*__{m}__*/state.json"):
            try:
                hs += [(h["epoch"], h["val"]) for h in json.loads(p.read_text())["history"]]
            except Exception:
                pass
        if hs:
            d = pd.DataFrame(hs, columns=["epoch", "val"]).groupby("epoch")["val"].mean()
            ax.plot(d.index, d.values, lw=1.3, label=m)
    ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("validation loss")
    ax.set_title("Baseline training curves (mean over runs)", loc="left")
    ax.legend(frameon=False, ncol=4, fontsize=7)
    fig.savefig(FIG / "nb03_fig2_curves.png"); plt.close(fig)
    print("  wrote nb03_fig2_curves.png")

    samp = sorted((WORK / "runs").glob("B_rva__*/preds_sample.npz"))
    if samp:
        fig, axes = plt.subplots(len(samp), 1, figsize=(11, 1.9 * len(samp)), sharex=True)
        axes = np.atleast_1d(axes)
        tt = np.arange(1024) / 128.0
        for ax, sp in zip(axes, samp):
            z = np.load(sp)
            k = min(3, len(z["y"]) - 1)
            ax.plot(tt, z["y"][k], lw=1.0, color=S["ink"], label="ground truth")
            ax.plot(tt, z["p"][k], lw=1.0, color=S["ecg"], label="reconstructed", alpha=.85)
            ax.set_ylabel(sp.parent.name.split("__")[1], rotation=0, ha="right",
                          va="center", fontsize=7.5)
            ax.tick_params(labelleft=False)
        axes[0].legend(frameon=False, ncol=2, fontsize=7)
        axes[0].set_title("Reconstructed vs. true ECG — one held-out window per baseline", loc="left")
        axes[-1].set_xlabel("seconds")
        fig.tight_layout(); fig.savefig(FIG / "nb03_fig3_qualitative.png"); plt.close(fig)
        print("  wrote nb03_fig3_qualitative.png")
MAJOR("04_figures")

In [ ]:
now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
card = f"""---
license: cc-by-4.0
tags: [radar, ecg, biosignals, contactless-monitoring, time-series]
pipeline_tag: audio-to-audio
---

# CardioMamba-Net — checkpoints and results

Radar-to-ECG reconstruction on the CR-RVS dataset. Produced by `03_baselines.ipynb` and
`04_cardiomamba_train.ipynb`. Corpus: [`{CFG['SRC_REPO']}`](https://huggingface.co/datasets/{CFG['SRC_REPO']}).

## Layout

```
runs/<experiment>__<model>__f<fold>/
    best.pt                  weights at the best validation epoch
    state.pt                 full resumable state (model, optimiser, scheduler, scaler, RNG)
    summary.json             config + aggregated test metrics
    metrics_windows.parquet  per-window metrics on held-out subjects
    metrics_subjects.parquet per-subject HR / HRV / peak-detection scores
    preds_sample.npz         200 held-out reconstructions for qualitative figures
results/                     cross-run tables, including ours vs. the published numbers
figures/                     comparison and training-curve figures
```

## Protocol

128 Hz, 1024-sample (8 s) windows. Splits **by subject**; test windows do not overlap.
Normalisation statistics are per fold from **training windows only**. Baselines use the paper's
single-channel input (`dy`), 5 levels, 64 base filters, plain MSE, Adam 5e-4.

Updated {now}.
"""
(WORK / "README.md").write_text(card)
ok = sync.flush(final=True, msg=f"{CFG['RUN_ID']} — {len(done)}/{len(QUEUE)} runs complete")
print("\n" + "=" * 76)
print("  SESSION COMPLETE" if ok else "  SESSION COMPLETE (final push had a problem)")
print("=" * 76)
print(f"  repo      : {sync.url}")
print(f"  runs done : {len(done)}/{len(QUEUE)}")
print(f"  this run  : {len(completed_now)} new")
print(f"  elapsed   : {(time.time()-t_start)/3600:.2f} h")
print("=" * 76)
if len(done) < len(QUEUE):
    print(f"\n  {len(QUEUE)-len(done)} run(s) remain. Start a NEW session and run this notebook")
    print("  again -- it resumes from Hugging Face and skips everything already finished.")
else:
    print("\n  Queue complete. Next: 04_cardiomamba_train.ipynb")

---
# 8 · Troubleshooting

**No GPU / very slow epochs** — *Session options → Accelerator → GPU T4 x2*. On CPU an epoch takes
minutes instead of seconds; the notebook runs but the queue will not finish.

**CUDA out of memory** — lower `CFG["BATCH"]` to 32 or 16. MultiResLinkNet with `BASE=64` is the
heaviest of the four. Batch 64 across two T4s is comfortable; batch 128 is not.

**`no normalisation stats for <exp>|<fold>`** — NB02 did not emit that combination, usually because
the split was empty after the quality gate. Re-run NB02, or drop that experiment from
`CFG["EXPERIMENTS"]`.

**`SUBJECT LEAK` assertion** — a genuine bug in fold assignment. Do not bypass it.

**Validation loss flat from epoch 1** — check `nb02_fig1_window.png`. If the ECG and `peak_map`
rows are not aligned, the targets are wrong and no architecture will help.

**DataLoader worker crashes** — set `CFG["WORKERS"] = 0`. Kaggle occasionally struggles with
worker processes plus memory-mapped `.npy` files.

**Session ended mid-queue** — expected. Start a new session, run the notebook again, and it
resumes. Progress is per run *and* per epoch.